# Chimera — Diagnostic iteration loop

Runs one Chimera training iteration end-to-end and inspects the structured outputs the workflow tracks.
This is the *canonical single-iteration entry point* — if you change a hyperparameter or an architectural detail, this notebook is what you run to verify the change.

**Pipeline**: data prep (idempotent) → train → per-epoch snapshot → final state dump → expectation check → summary JSON → diagnostic plots.

See `training/workflow.md` for the pre-registered expectations (E1–E7) and the iteration protocol.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = Path('..').resolve()
RUNS = REPO / 'training' / 'runs'
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'training'))

# Best known config for close-species (Human+Monkey). For a fresh dataset,
# start here and adjust based on expectation failures. See workflow.md for
# the first-suspect table.
RUN_NAME = 'demo-iteration'
CONFIG = dict(
    run_name=RUN_NAME,
    adata='data/processed/camex_human_monkey.h5ad',
    max_epochs=100,
    config='D',
    adv_weight=100.0,
    lambda_fit=0.1,
    lambda_anchor=0.001,
    lambda_res=0.01,
    warmup_adversarial_epochs=5,
    warmup_fqr_epochs=20,
    warmup_kl_epochs=20,
    accelerator='mps',
    snapshot_every_n_epochs=5,
    seed=42,
)

## 1. Build the 2-species subset (idempotent)

Skipped if the file already exists. Change `--species` to rebuild a different subset.

In [ ]:
subset = REPO / CONFIG['adata']
if not subset.exists():
    subprocess.run([sys.executable, str(REPO/'training/build_subset.py'),
                    '--species', 'Human', 'Monkey',
                    '--dest', str(subset)], cwd=REPO, check=True)
print('Subset present:', subset, subset.exists())

## 2. Run training + diagnostics

`run_experiment.py` wraps `train_and_diagnose.py` + `check_expectations.py` and writes a single `summary.json`.
For a fresh iteration, uncomment the subprocess call below. For inspecting an existing run, leave it commented.

In [ ]:
# args = [sys.executable, str(REPO/'training/run_experiment.py')]
# for k, v in CONFIG.items():
#     if isinstance(v, bool):
#         if v: args.append(f'--{k.replace("_", "-")}')
#     else:
#         args += [f'--{k.replace("_", "-")}', str(v)]
# subprocess.run(args, cwd=REPO, check=True)

RUN_DIR = RUNS / RUN_NAME
assert RUN_DIR.exists(), f'{RUN_DIR} missing — uncomment the cell above to run.'
summary = json.loads((RUN_DIR / 'summary.json').read_text())
print(f"{summary['run_name']}: {summary['expectation_summary']['pass']}/"
      f"{sum(summary['expectation_summary'].values()) if isinstance(summary['expectation_summary'].values(), dict) else 28} pass")
print('Failed:', summary['failed_expectations'])

## 3. Diagnostic plots

Six canonical plots (no scib — fast, per-epoch).

In [ ]:
subprocess.run([sys.executable, str(REPO/'training/plot_diagnostics.py'),
                '--run-name', RUN_NAME], cwd=REPO, check=True)
plots = sorted((RUN_DIR/'plots').glob('*.png'))
print('Plots available:', [p.name for p in plots])

In [ ]:
# Show them inline
from IPython.display import Image, display
for p in plots:
    print(p.name)
    display(Image(str(p)))

## 4. Structured expectation report

In [ ]:
print((RUN_DIR/'report.md').read_text())

## 5. Deep-dive: per-gene alpha trajectory

For diagnosing FM1 (alpha → 0 universally) and FM4 (Δα one-sided).

In [ ]:
snap_dir = RUN_DIR / 'alpha_snapshots'
snaps = sorted(snap_dir.glob('alpha_epoch_*.npz'))
if snaps:
    epochs = [int(p.stem.split('_')[-1]) for p in snaps]
    alphas = np.stack([np.load(p)['alpha'] for p in snaps])
    prior = np.load(snaps[0])['prior']
    deltas = alphas - prior[None, :]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    # trajectory of 20 representative genes (extremes + middle)
    order = np.argsort(prior)
    sampled = np.concatenate([order[:5], order[len(order)//2-2:len(order)//2+3], order[-5:]])
    for g in sampled:
        ax[0].plot(epochs, alphas[:, g], alpha=0.6)
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('alpha')
    ax[0].set_title('Alpha trajectory (representative genes)')
    ax[0].grid(alpha=0.3)
    # Δα mean / std / q05 / q95 over epochs
    ax[1].plot(epochs, deltas.mean(axis=1), label='mean Δα')
    ax[1].fill_between(epochs, np.quantile(deltas, 0.05, axis=1),
                       np.quantile(deltas, 0.95, axis=1), alpha=0.2, label='5–95% Δα')
    ax[1].axhline(0, color='k', lw=0.5)
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('Δα'); ax[1].legend()
    ax[1].set_title('Δα distribution over training')
    ax[1].grid(alpha=0.3)
    plt.tight_layout()

## 6. Decide next iteration

Look at `report.md` failures. For each, check the first-suspect table in `workflow.md`. Then:

- Change **one** thing at a time so failures remain attributable.
- Record the change's rationale in `iteration_log.md`.
- Re-run this notebook with a new `RUN_NAME`.

Compare across runs:

In [ ]:
out = subprocess.run([sys.executable, str(REPO/'training/summarize_runs.py')],
                     cwd=REPO, capture_output=True, text=True)
print(out.stdout)